# MOSAIC End-to-End Runtime Rerun

This notebook measures full prefill, ideal prefix reuse, materialized cache branching, and cold compilation on TinyLlama, Qwen, and Mistral architectures. Wall-clock timing uses `perf_counter_ns()` with CUDA synchronization so host-side branch costs are included.

In [ ]:
import os
import sys
import subprocess

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q", "--upgrade",
    "transformers>=4.57,<5",
    "accelerate>=1.10,<2",
    "bitsandbytes>=0.46,<1",
    "huggingface_hub>=0.34,<1",
    "safetensors>=0.6,<1",
    "sentencepiece>=0.2,<1",
    "pandas>=2.2,<3",
    "matplotlib>=3.8,<4",
])

In [ ]:
import gc
import hashlib
import inspect
import json
import math
import platform
import time
import traceback
import warnings
import zipfile
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import transformers
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    DynamicCache,
)

warnings.filterwarnings("ignore")
torch.set_float32_matmul_precision("high")

ROOT = Path("/kaggle/working/mosaic_llm_e2e_rerun")
RESULTS = ROOT / "results"
FIGURES = ROOT / "figures"
RESULTS.mkdir(parents=True, exist_ok=True)
FIGURES.mkdir(parents=True, exist_ok=True)

if not torch.cuda.is_available():
    raise RuntimeError("Enable a Kaggle GPU accelerator.")

DEVICE = torch.device("cuda:0")
GPU_NAME = torch.cuda.get_device_name(0)
GPU_BYTES = torch.cuda.get_device_properties(0).total_memory
print("GPU:", GPU_NAME, GPU_BYTES / 2**30, "GiB")
print("torch:", torch.__version__)
print("transformers:", transformers.__version__)

In [ ]:
PROFILE = "paper_t4"

PROFILES = {
    "smoke": {
        "contexts": [256, 1024],
        "repeats": 3,
        "warmups": 1,
        "suffixes": 1,
        "stream_requests": 8,
    },
    "paper_t4": {
        "contexts": [256, 512, 1024, 2048],
        "repeats": 10,
        "warmups": 2,
        "suffixes": 3,
        "stream_requests": 16,
    },
}

CFG = PROFILES[PROFILE]

MODELS = [
    {
        "family": "Llama-architecture",
        "model_id": "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
        "quantization": None,
    },
    {
        "family": "Qwen",
        "model_id": "Qwen/Qwen2.5-1.5B-Instruct",
        "quantization": None,
    },
    {
        "family": "Mistral",
        "model_id": "mistralai/Mistral-7B-Instruct-v0.3",
        "quantization": "bnb4",
    },
]

CONTEXTS = CFG["contexts"]
REPEATS = CFG["repeats"]
WARMUPS = CFG["warmups"]
STREAM_REQUESTS = CFG["stream_requests"]

In [ ]:
SYSTEM = (
    "You are a careful repository and retrieval analysis agent. "
    "Preserve exact symbol names, cite paths, never invent execution results, "
    "and distinguish verified facts from hypotheses."
)

TOOLS = json.dumps({
    "tools": [
        {
            "name": "search_repository",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {"type": "string"},
                    "path": {"type": "string"},
                },
                "required": ["query"],
            },
        },
        {
            "name": "run_tests",
            "parameters": {
                "type": "object",
                "properties": {
                    "target": {"type": "string"},
                },
                "required": ["target"],
            },
        },
    ]
}, indent=2)

RAG = "\n\n".join([
    "Prefix KV caching is valid only when requests share an exact causal prefix.",
    "Cache identity includes model revision, tokenizer, and exact token content.",
    "Mutable request branches must not alter an immutable shared prefix.",
    "End-to-end latency includes lookup, branch creation, scheduling, and GPU execution.",
] * 128)

PREFIX_TEXTS = {
    "agent": SYSTEM + "\n\nTOOLS:\n" + TOOLS,
    "rag": SYSTEM + "\n\nRETRIEVED DOCUMENTS:\n" + RAG,
}

SUFFIX_TEXTS = [
    "\nUser: Explain cache invalidation.\nAssistant:",
    "\nUser: Propose branch-isolation tests.\nAssistant:",
    "\nUser: Estimate the compilation break-even.\nAssistant:",
][:CFG["suffixes"]]

In [ ]:
def sync():
    torch.cuda.synchronize()

def clear_gpu():
    gc.collect()
    torch.cuda.empty_cache()
    sync()

def filtered_kwargs(model, kwargs):
    signature = inspect.signature(model.forward)
    return {
        key: value for key, value in kwargs.items()
        if key in signature.parameters and value is not None
    }

def model_device(model):
    return next(model.parameters()).device

def maximum_context(model):
    values = []
    for name in ("max_position_embeddings", "n_positions", "max_sequence_length"):
        value = getattr(model.config, name, None)
        if isinstance(value, int) and value > 0:
            values.append(value)
    return min(values) if values else 4096

def make_prefix(tokenizer, text, target):
    unit = tokenizer.encode(text, add_special_tokens=False)
    if not unit:
        raise ValueError("Empty prefix.")
    return (unit * math.ceil(target / len(unit)))[:target]

def make_suffix(tokenizer, text, limit=32):
    ids = tokenizer.encode(text, add_special_tokens=False)[:limit]
    if ids:
        return ids
    if tokenizer.eos_token_id is None:
        raise ValueError("Empty suffix and no EOS token.")
    return [tokenizer.eos_token_id]

def cache_key(model_id, revision, tokenizer, ids):
    raw = json.dumps({
        "model_id": model_id,
        "revision": revision,
        "tokenizer": tokenizer.__class__.__name__,
        "tokens": [int(x) for x in ids],
    }, sort_keys=True, separators=(",", ":")).encode()
    return hashlib.sha256(raw).hexdigest()

In [ ]:
def freeze_cache(past_key_values):
    legacy = (
        past_key_values.to_legacy_cache()
        if hasattr(past_key_values, "to_legacy_cache")
        else past_key_values
    )
    return tuple(
        tuple(t.detach().clone() for t in layer)
        for layer in legacy
    )

def fresh_cache(model, frozen):
    try:
        return DynamicCache.from_legacy_cache(frozen)
    except Exception:
        try:
            cache = DynamicCache(config=model.config)
        except TypeError:
            cache = DynamicCache()
        for layer_index, layer in enumerate(frozen):
            cache.update(layer[0], layer[1], layer_index)
        return cache

def frozen_cache_bytes(frozen):
    seen = set()
    total = 0
    for layer in frozen:
        for tensor in layer:
            pointer = tensor.data_ptr()
            if pointer not in seen:
                seen.add(pointer)
                total += tensor.numel() * tensor.element_size()
    return int(total)

def logit_metrics(full_logits, cached_logits):
    full = full_logits.float()
    cached = cached_logits.float()
    delta = full - cached
    full_logp = F.log_softmax(full, dim=-1)
    cached_logp = F.log_softmax(cached, dim=-1)
    kl = (full_logp.exp() * (full_logp - cached_logp)).sum(-1).mean()
    return {
        "max_abs_logit_error": float(delta.abs().max()),
        "mean_abs_logit_error": float(delta.abs().mean()),
        "cosine_similarity": float(F.cosine_similarity(full, cached, dim=-1).mean()),
        "top1_agreement": float((full.argmax(-1) == cached.argmax(-1)).float().mean()),
        "kl_full_to_cached": float(kl),
    }

In [ ]:
def load_model(spec):
    tokenizer = AutoTokenizer.from_pretrained(spec["model_id"], use_fast=True)
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token

    kwargs = {
        "device_map": {"": 0},
        "low_cpu_mem_usage": True,
    }
    if spec["quantization"] == "bnb4":
        kwargs["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True,
        )
    else:
        kwargs["dtype"] = torch.float16

    try:
        model = AutoModelForCausalLM.from_pretrained(
            spec["model_id"],
            attn_implementation="sdpa",
            **kwargs,
        )
    except Exception:
        model = AutoModelForCausalLM.from_pretrained(spec["model_id"], **kwargs)

    model.eval()
    revision = getattr(model.config, "_commit_hash", None) or "unknown"
    return tokenizer, model, revision

In [ ]:
def gpu_time_ms(fn, repeats, warmups):
    for _ in range(warmups):
        output = fn()
        del output
    sync()

    values = []
    for _ in range(repeats):
        start = torch.cuda.Event(enable_timing=True)
        end = torch.cuda.Event(enable_timing=True)
        start.record()
        output = fn()
        end.record()
        sync()
        values.append(float(start.elapsed_time(end)))
        del output
    return values

def wall_time_ms(fn, repeats, warmups):
    for _ in range(warmups):
        output = fn()
        sync()
        del output

    values = []
    for _ in range(repeats):
        sync()
        start = time.perf_counter_ns()
        output = fn()
        sync()
        end = time.perf_counter_ns()
        values.append((end - start) / 1e6)
        del output
    return values

def median(values):
    return float(np.median(np.asarray(values, dtype=np.float64)))

In [ ]:
def prepare_tensors(model, prefix_ids, suffix_ids):
    device = model_device(model)
    prefix = torch.tensor(prefix_ids, dtype=torch.long, device=device).unsqueeze(0)
    suffix = torch.tensor(suffix_ids, dtype=torch.long, device=device).unsqueeze(0)
    full = torch.cat([prefix, suffix], dim=1)

    full_positions = torch.arange(full.shape[1], device=device).unsqueeze(0)
    suffix_positions = torch.arange(
        prefix.shape[1], full.shape[1], device=device
    ).unsqueeze(0)

    return {
        "prefix": prefix,
        "prefix_mask": torch.ones_like(prefix),
        "prefix_positions": torch.arange(prefix.shape[1], device=device).unsqueeze(0),
        "suffix": suffix,
        "suffix_mask": torch.ones((1, full.shape[1]), dtype=torch.long, device=device),
        "suffix_positions": suffix_positions,
        "full": full,
        "full_mask": torch.ones_like(full),
        "full_positions": full_positions,
    }

@torch.inference_mode()
def compile_prefix(model, tensors):
    kwargs = filtered_kwargs(model, {
        "input_ids": tensors["prefix"],
        "attention_mask": tensors["prefix_mask"],
        "position_ids": tensors["prefix_positions"],
        "cache_position": tensors["prefix_positions"][0],
        "use_cache": True,
    })

    sync()
    start = time.perf_counter_ns()
    output = model(**kwargs)
    frozen = freeze_cache(output.past_key_values)
    sync()
    elapsed_ms = (time.perf_counter_ns() - start) / 1e6
    return frozen, float(elapsed_ms)

def full_call(model, tensors):
    kwargs = filtered_kwargs(model, {
        "input_ids": tensors["full"],
        "attention_mask": tensors["full_mask"],
        "position_ids": tensors["full_positions"],
        "cache_position": tensors["full_positions"][0],
        "use_cache": True,
    })
    return model(**kwargs).logits[:, -1, :]

def cached_call(model, tensors, cache):
    kwargs = filtered_kwargs(model, {
        "input_ids": tensors["suffix"],
        "attention_mask": tensors["suffix_mask"],
        "position_ids": tensors["suffix_positions"],
        "cache_position": tensors["suffix_positions"][0],
        "past_key_values": cache,
        "use_cache": True,
    })
    return model(**kwargs).logits[:, -1, :]

In [ ]:
def benchmark_case(model, tokenizer, model_id, revision, prefix_ids, suffix_ids):
    tensors = prepare_tensors(model, prefix_ids, suffix_ids)
    frozen, compile_wall_ms = compile_prefix(model, tensors)

    full_logits = full_call(model, tensors)
    cached_logits = cached_call(model, tensors, fresh_cache(model, frozen))
    metrics = logit_metrics(full_logits, cached_logits)

    full_gpu = gpu_time_ms(
        lambda: full_call(model, tensors), REPEATS, WARMUPS
    )
    full_wall = wall_time_ms(
        lambda: full_call(model, tensors), REPEATS, WARMUPS
    )

    # Ideal: create a fresh branch outside each timed interval.
    for _ in range(WARMUPS):
        branch = fresh_cache(model, frozen)
        output = cached_call(model, tensors, branch)
        sync()
        del output, branch

    ideal_gpu = []
    ideal_wall = []
    branch_only = []
    for _ in range(REPEATS):
        branch_start = time.perf_counter_ns()
        branch = fresh_cache(model, frozen)
        branch_only.append((time.perf_counter_ns() - branch_start) / 1e6)

        start_event = torch.cuda.Event(enable_timing=True)
        end_event = torch.cuda.Event(enable_timing=True)
        sync()
        wall_start = time.perf_counter_ns()
        start_event.record()
        output = cached_call(model, tensors, branch)
        end_event.record()
        sync()
        wall_end = time.perf_counter_ns()
        ideal_gpu.append(float(start_event.elapsed_time(end_event)))
        ideal_wall.append((wall_end - wall_start) / 1e6)
        del output, branch

    # Materialized: branch creation is inside the wall-clock interval.
    materialized_gpu = []
    materialized_wall = []
    for _ in range(WARMUPS):
        branch = fresh_cache(model, frozen)
        output = cached_call(model, tensors, branch)
        sync()
        del output, branch

    for _ in range(REPEATS):
        sync()
        start_event = torch.cuda.Event(enable_timing=True)
        end_event = torch.cuda.Event(enable_timing=True)
        wall_start = time.perf_counter_ns()
        branch = fresh_cache(model, frozen)
        start_event.record()
        output = cached_call(model, tensors, branch)
        end_event.record()
        sync()
        wall_end = time.perf_counter_ns()
        materialized_gpu.append(float(start_event.elapsed_time(end_event)))
        materialized_wall.append((wall_end - wall_start) / 1e6)
        del output, branch

    full_wall_median = median(full_wall)
    ideal_wall_median = median(ideal_wall)
    materialized_wall_median = median(materialized_wall)

    baseline_stream = STREAM_REQUESTS * full_wall_median
    ideal_stream = compile_wall_ms + STREAM_REQUESTS * ideal_wall_median
    materialized_stream = (
        compile_wall_ms + STREAM_REQUESTS * materialized_wall_median
    )

    ideal_saving = full_wall_median - ideal_wall_median
    materialized_saving = full_wall_median - materialized_wall_median

    changed = list(prefix_ids)
    changed[len(changed) // 2] = (
        changed[len(changed) // 2] + 1
    ) % max(2, tokenizer.vocab_size)
    key = cache_key(model_id, revision, tokenizer, prefix_ids)

    result = {
        "prefix_tokens": len(prefix_ids),
        "suffix_tokens": len(suffix_ids),
        "compile_wall_ms": compile_wall_ms,
        "cache_bytes": frozen_cache_bytes(frozen),
        "full_gpu_ms": median(full_gpu),
        "full_wall_ms": full_wall_median,
        "ideal_gpu_ms": median(ideal_gpu),
        "ideal_wall_ms": ideal_wall_median,
        "materialized_gpu_ms": median(materialized_gpu),
        "materialized_wall_ms": materialized_wall_median,
        "branch_creation_ms": median(branch_only),
        "ideal_gpu_speedup": median(full_gpu) / median(ideal_gpu),
        "ideal_wall_speedup": full_wall_median / ideal_wall_median,
        "materialized_wall_speedup": (
            full_wall_median / materialized_wall_median
        ),
        "ideal_stream_speedup": baseline_stream / ideal_stream,
        "materialized_stream_speedup": (
            baseline_stream / materialized_stream
        ),
        "ideal_break_even_requests": (
            compile_wall_ms / ideal_saving
            if ideal_saving > 0 else float("inf")
        ),
        "materialized_break_even_requests": (
            compile_wall_ms / materialized_saving
            if materialized_saving > 0 else float("inf")
        ),
        "exact_key_hit": (
            cache_key(model_id, revision, tokenizer, prefix_ids) == key
        ),
        "changed_key_hit": (
            cache_key(model_id, revision, tokenizer, changed) == key
        ),
        **metrics,
        "full_wall_raw_ms": json.dumps(full_wall),
        "ideal_wall_raw_ms": json.dumps(ideal_wall),
        "materialized_wall_raw_ms": json.dumps(materialized_wall),
    }

    result["correctness_pass"] = (
        result["top1_agreement"] == 1.0
        and result["cosine_similarity"] > 0.999
        and result["kl_full_to_cached"] < 1e-3
    )
    result["cache_key_safety_pass"] = (
        result["exact_key_hit"] and not result["changed_key_hit"]
    )
    return result

In [ ]:
rows = []
failures = []

for spec in MODELS:
    tokenizer = None
    model = None
    try:
        clear_gpu()
        print("\nLoading:", spec["family"], spec["model_id"])
        tokenizer, model, revision = load_model(spec)
        max_context = maximum_context(model)

        for workload, prefix_text in PREFIX_TEXTS.items():
            suffix_lists = [
                make_suffix(tokenizer, text) for text in SUFFIX_TEXTS
            ]
            maximum_suffix = max(map(len, suffix_lists))

            for requested_context in CONTEXTS:
                prefix_target = min(
                    requested_context,
                    max_context - maximum_suffix - 1,
                )
                if prefix_target < 32:
                    continue

                prefix_ids = make_prefix(
                    tokenizer, prefix_text, prefix_target
                )

                for suffix_index, suffix_ids in enumerate(suffix_lists):
                    row = benchmark_case(
                        model=model,
                        tokenizer=tokenizer,
                        model_id=spec["model_id"],
                        revision=revision,
                        prefix_ids=prefix_ids,
                        suffix_ids=suffix_ids,
                    )
                    row.update({
                        "family": spec["family"],
                        "model_id": spec["model_id"],
                        "revision": revision,
                        "quantization": spec["quantization"] or "fp16",
                        "workload": workload,
                        "requested_context": requested_context,
                        "suffix_index": suffix_index,
                        "gpu": GPU_NAME,
                    })
                    rows.append(row)
                    print(
                        spec["family"],
                        workload,
                        requested_context,
                        f"wall={row['materialized_wall_speedup']:.2f}x",
                        f"stream={row['materialized_stream_speedup']:.2f}x",
                        f"branch={row['branch_creation_ms']:.3f} ms",
                        "correct=", row["correctness_pass"],
                    )

                clear_gpu()

    except Exception as error:
        failures.append({
            "family": spec["family"],
            "model_id": spec["model_id"],
            "error_type": type(error).__name__,
            "error": repr(error),
            "traceback": traceback.format_exc(),
        })
        print("FAILED:", repr(error))
    finally:
        if model is not None:
            del model
        if tokenizer is not None:
            del tokenizer
        clear_gpu()

results_df = pd.DataFrame(rows)
failures_df = pd.DataFrame(failures)
results_df.to_csv(RESULTS / "e2e_runtime_results.csv", index=False)
failures_df.to_csv(RESULTS / "failures.csv", index=False)
display(results_df.head())
display(failures_df)

In [ ]:
summary_rows = []

if not results_df.empty:
    for family, group in results_df.groupby("family"):
        long_group = group[group["prefix_tokens"] >= 1024]
        if long_group.empty:
            long_group = group

        summary_rows.append({
            "family": family,
            "rows": len(group),
            "correctness_rate": group["correctness_pass"].mean(),
            "cache_key_safety_rate": group["cache_key_safety_pass"].mean(),
            "median_materialized_wall_speedup": (
                long_group["materialized_wall_speedup"].median()
            ),
            "minimum_materialized_wall_speedup": (
                long_group["materialized_wall_speedup"].min()
            ),
            "positive_long_context_rate": (
                long_group["materialized_wall_speedup"] > 1.0
            ).mean(),
            "median_16_request_stream_speedup": (
                long_group["materialized_stream_speedup"].median()
            ),
            "median_branch_creation_ms": (
                long_group["branch_creation_ms"].median()
            ),
            "median_break_even_requests": (
                long_group["materialized_break_even_requests"].median()
            ),
        })

summary_df = pd.DataFrame(summary_rows)
if not summary_df.empty:
    summary_df["consistent_gain_pass"] = (
        (summary_df["correctness_rate"] == 1.0)
        & (summary_df["cache_key_safety_rate"] == 1.0)
        & (summary_df["positive_long_context_rate"] == 1.0)
        & (summary_df["median_16_request_stream_speedup"] > 1.05)
    )

summary_df.to_csv(RESULTS / "e2e_consistency_summary.csv", index=False)
display(summary_df)

In [ ]:
if not results_df.empty:
    plot_df = (
        results_df.groupby(["family", "prefix_tokens"], as_index=False)
        .agg(
            wall_speedup=("materialized_wall_speedup", "median"),
            stream_speedup=("materialized_stream_speedup", "median"),
            cache_bytes=("cache_bytes", "median"),
        )
    )

    for family, group in plot_df.groupby("family"):
        plt.plot(
            group["prefix_tokens"],
            group["wall_speedup"],
            marker="o",
            label=family,
        )
    plt.axhline(1.0, linestyle="--")
    plt.xscale("log", base=2)
    plt.xlabel("Exact shared-prefix tokens")
    plt.ylabel("Synchronized wall-clock speedup")
    plt.title("End-to-end prefix-reuse latency")
    plt.legend()
    plt.tight_layout()
    plt.savefig(FIGURES / "e2e_wall_speedup.png", dpi=200)
    plt.show()

    for family, group in plot_df.groupby("family"):
        plt.plot(
            group["prefix_tokens"],
            group["stream_speedup"],
            marker="o",
            label=family,
        )
    plt.axhline(1.0, linestyle="--")
    plt.xscale("log", base=2)
    plt.xlabel("Exact shared-prefix tokens")
    plt.ylabel(f"{STREAM_REQUESTS}-request amortized speedup")
    plt.title("Cold compile cost included")
    plt.legend()
    plt.tight_layout()
    plt.savefig(FIGURES / "e2e_stream_speedup.png", dpi=200)
    plt.show()

## External serving baseline

For a production comparison, run the same request stream with vLLM Automatic Prefix Caching disabled and enabled, then compare time to first token and request throughput.

In [ ]:
environment = {
    "completed_utc": pd.Timestamp.utcnow().isoformat(),
    "profile": PROFILE,
    "python": sys.version,
    "platform": platform.platform(),
    "torch": torch.__version__,
    "transformers": transformers.__version__,
    "cuda_runtime": torch.version.cuda,
    "gpu": GPU_NAME,
    "gpu_memory_bytes": GPU_BYTES,
    "repeats": REPEATS,
    "warmups": WARMUPS,
    "stream_requests": STREAM_REQUESTS,
    "models": MODELS,
}
(RESULTS / "environment.json").write_text(
    json.dumps(environment, indent=2),
    encoding="utf-8",
)

verdict = {
    "requested_families": [model["family"] for model in MODELS],
    "completed_families": (
        sorted(results_df["family"].unique().tolist())
        if not results_df.empty else []
    ),
    "all_completed": (
        not results_df.empty
        and set(results_df["family"]) == {
            model["family"] for model in MODELS
        }
    ),
    "all_consistent": (
        not summary_df.empty
        and bool(summary_df["consistent_gain_pass"].all())
    ),
}
verdict["global_e2e_consistency_pass"] = (
    verdict["all_completed"] and verdict["all_consistent"]
)
(RESULTS / "global_e2e_verdict.json").write_text(
    json.dumps(verdict, indent=2),
    encoding="utf-8",
)
print(json.dumps(verdict, indent=2))

zip_path = Path("/kaggle/working/mosaic_llm_e2e_rerun_results.zip")
if zip_path.exists():
    zip_path.unlink()

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as archive:
    for file in ROOT.rglob("*"):
        if file.is_file():
            archive.write(file, file.relative_to(ROOT))

print("Upload:")
print("1.", zip_path)
print("2. this executed notebook")